# SKHynix PBL 시계열 시퀀스 모델링 (5주차)


## 📌 개요

시간대(timekey_hr) 내에서 공정 순서(oper_id)를 고려한 시퀀스 기반 TAT 예측 모델입니다. 동일한 timekey_hr 내의 oper_id들을 순서대로 정렬하여 시퀀스 데이터로 구성하고, 각 oper별 개별 예측(sequence-to-sequence)을 수행합니다.

**데이터 구조**: `[batch_size, sequence_length, feature_dim]`
- **sequence_length**: timekey_hr 내 함께 구성될 oper_id 개수 + 가장 많은 group의 oper_id 수
- **feature_dim**: 연속형 변수 개수 + 범주형 변수 개수 × 임베딩 차원

## 🔧 환경 설정 및 라이브러리

In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import yaml
import logging
import json
import argparse
import math

from datetime import datetime
from tqdm import tqdm
from typing import Dict, List, Tuple, Optional, Union
from types import SimpleNamespace
from collections import defaultdict

# sklearn
from sklearn.preprocessing import LabelEncoder, RobustScaler, StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

# PyTorch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torch.optim import Adam, AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau, StepLR

# Oper Transformer

## 📊 유틸리티 함수들

### 설정 로딩 및 시드 설정

In [2]:
def load_config(config_dir: str = "configs") -> Dict:
    """YAML 설정 파일들을 통합하여 로드"""
    configs = {}
    config_files = ["dataset", "model", "training"]

    for file in config_files:
        config_path = os.path.join(config_dir, f"{file}.yaml")
        with open(config_path, "r", encoding="utf-8") as f:
            config = yaml.safe_load(f)
            configs.update(config)

    return configs


def set_random_seeds(seed: int = 42):
    """재현성을 위한 랜덤 시드 설정"""
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def setup_logging(log_file: str = "training.log"):
    """로깅 설정"""
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s - %(levelname)s - %(message)s",
        handlers=[
            logging.FileHandler(log_file),
            logging.StreamHandler()
        ]
    )
    return logging.getLogger(__name__)

logger = setup_logging()

### 범주형 데이터 처리기

In [3]:
class CategoricalProcessor:
    """범주형 변수 임베딩을 위한 처리기"""
    
    def __init__(self, embedding_dim: int = 8):
        self.embedding_dim = embedding_dim
        self.label_encoders = {}
        self.vocab_sizes = {}
        self.categorical_columns = []
        
    def fit(self, df: pd.DataFrame, categorical_columns: List[str]):
        """전체 데이터에 대해 범주형 인코더 학습"""
        self.categorical_columns = categorical_columns
        
        for col in categorical_columns:
            unique_values = df[col].astype(str).unique()
            encoder = LabelEncoder()
            encoder.fit(unique_values)
            
            self.label_encoders[col] = encoder
            self.vocab_sizes[col] = len(encoder.classes_)
        
        logger.info(f"범주형 변수별 고유값 개수:")
        for col in categorical_columns:
            logger.info(f"  {col}: {self.vocab_sizes[col]}개")
    
    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """DataFrame의 범주형 컬럼들을 숫자로 변환"""
        df_encoded = df.copy()
        
        for col in self.categorical_columns:
            df_encoded[col] = self.label_encoders[col].transform(
                df_encoded[col].astype(str)
            )
        
        return df_encoded
    
    def get_vocab_sizes(self) -> List[int]:
        """각 범주형 변수의 vocab_size 리스트 반환"""
        return [self.vocab_sizes[col] for col in self.categorical_columns]

## 🗂️ 시퀀스 데이터셋 클래스

### 메인 데이터셋

In [4]:
class GroupedOperDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,  # 입력 데이터프레임
        categorical_columns: List[str],  # 범주형 특성 컬럼 리스트
        continuous_columns: List[str],  # 연속형 특성 컬럼 리스트
        target_column: str = "y",  # 타겟(예측 대상) 컬럼명
        group_map: Dict = None,  # 그룹 매핑 정보 (옵션)
        window_size: int = 5,  # 슬라이딩 윈도우 크기
        window_stride: int = 1,  # 윈도우 이동 간격
        padding_value: float = 0.0,  # 패딩에 사용할 값
        group_position: str = 'middle'  # 그룹 결정 기준 위치 ('first', 'middle', 'last')
    ):
        # 클래스 속성으로 각 파라미터 저장
        self.categorical_columns = categorical_columns
        self.continuous_columns = continuous_columns
        self.target_column = target_column
        self.group_map = group_map
        self.window_size = window_size
        self.window_stride = window_stride
        self.padding_value = padding_value
        self.group_position = group_position
        
        # group_position 파라미터 유효성 검사
        if group_position not in ['first', 'middle', 'last']:
            raise ValueError(f"group_position must be 'first', 'middle', or 'last', got {group_position}")
        
        # 특성 차원 계산
        self.continuous_dim = len(continuous_columns)  # 연속형 특성의 개수
        self.categorical_dim = len(categorical_columns)  # 범주형 특성의 개수
        
        # 데이터 전처리 (최적화)
        self._preprocess_data_optimized(df)  # 최적화된 전처리 함수 호출
        
        # 시퀀스 생성 (최적화)
        self._create_sequences_optimized()  # 최적화된 시퀀스 생성 함수 호출
        
        # 데이터셋 생성 정보 로깅
        logger.info(f"GroupedOperDataset 생성 완료:")
        logger.info(f"  - 총 시퀀스 수: {len(self.sequences)}")
        logger.info(f"  - Window 크기: {window_size}")
        logger.info(f"  - Stride: {window_stride}")
        logger.info(f"  - 전체 시퀀스 길이: {self.total_sequence_length}")
        logger.info(f"  - 최대 그룹 크기: {self.max_group_size}")
        logger.info(f"  - 그룹 기준 위치: {group_position}")
    
    def _preprocess_data_optimized(self, df):
        """최적화된 데이터 전처리"""
        # oper_num 추출 (벡터화) - 'oper' 접두사 제거하고 숫자만 추출
        oper_nums = df['oper_id'].str.slice(4).astype(int).values
        
        # NumPy 배열로 정렬 (DataFrame 정렬보다 빠름)
        timekeys = df['timekey_hr'].values  # 시간 키 배열
        # lexsort: 다중 키 정렬 (oper_num으로 먼저 정렬 후 timekey로 정렬)
        sort_indices = np.lexsort((oper_nums, timekeys))
        
        # 정렬된 데이터를 NumPy 배열로 직접 저장
        self.timekeys = timekeys[sort_indices]  # 정렬된 시간 키
        self.oper_nums = oper_nums[sort_indices]  # 정렬된 공정 번호
        self.oper_ids = df['oper_id'].values[sort_indices]  # 정렬된 공정 ID
        self.oper_groups = df['oper_group'].values[sort_indices]  # 정렬된 공정 그룹
        
        # 특성 데이터를 NumPy 배열로 (메모리 연속성)
        # 연속형 특성 배열 (float32로 변환하여 메모리 절약)
        self.continuous_array = df[self.continuous_columns].values[sort_indices].astype(np.float32)
        if self.categorical_columns:  # 범주형 컬럼이 있는 경우
            # 범주형 특성 배열
            self.categorical_array = df[self.categorical_columns].values[sort_indices].astype(np.float32)
        else:  # 범주형 컬럼이 없는 경우
            # 빈 배열 생성
            self.categorical_array = np.zeros((len(df), 0), dtype=np.float32)
        # 타겟 배열
        self.target_array = df[self.target_column].values[sort_indices].astype(np.float32)
        
        # 최대 그룹 크기 계산 (벡터화)
        unique_timekeys = np.unique(self.timekeys)  # 고유한 시간 키
        max_group_size = 0  # 최대 그룹 크기 초기화
        for tk in unique_timekeys:  # 각 시간 키에 대해
            tk_mask = self.timekeys == tk  # 현재 시간 키에 해당하는 마스크
            tk_groups = self.oper_groups[tk_mask]  # 해당 시간의 그룹들
            unique_groups, counts = np.unique(tk_groups, return_counts=True)  # 그룹별 개수 계산
            max_group_size = max(max_group_size, counts.max())  # 최대값 업데이트
        self.max_group_size = max_group_size  # 최대 그룹 크기 저장
        
        # 전체 시퀀스 길이 = 윈도우 크기 + 최대 그룹 크기
        self.total_sequence_length = self.window_size + self.max_group_size
        
        # 빠른 인덱싱을 위한 구조 (NumPy 기반)
        self.timekey_starts = {}  # 각 시간 키의 시작 인덱스
        self.timekey_ends = {}  # 각 시간 키의 끝 인덱스
        current_tk = self.timekeys[0]  # 첫 번째 시간 키
        start_idx = 0  # 시작 인덱스
        
        # 시간 키별 인덱스 범위 계산
        for i in range(1, len(self.timekeys)):
            if self.timekeys[i] != current_tk:  # 시간 키가 변경되면
                self.timekey_starts[current_tk] = start_idx  # 현재 키의 시작 인덱스 저장
                self.timekey_ends[current_tk] = i  # 현재 키의 끝 인덱스 저장
                current_tk = self.timekeys[i]  # 새로운 시간 키로 업데이트
                start_idx = i  # 새로운 시작 인덱스
        # 마지막 시간 키 처리
        self.timekey_starts[current_tk] = start_idx
        self.timekey_ends[current_tk] = len(self.timekeys)
    
    def _get_group_reference_index(self, window_size: int) -> int:
        """그룹을 결정할 참조 공정의 인덱스 반환"""
        if self.group_position == 'first':
            return 0  # 윈도우의 첫 번째 공정을 기준으로
        elif self.group_position == 'middle':
            return window_size // 2  # 윈도우의 중간 공정을 기준으로
        elif self.group_position == 'last':
            return window_size - 1  # 윈도우의 마지막 공정을 기준으로
    
    def _create_sequences_optimized(self):
        """최적화된 시퀀스 생성"""
        self.sequences = []  # 생성된 시퀀스를 저장할 리스트 초기화
        ref_idx = self._get_group_reference_index(self.window_size)  # 참조 인덱스 계산
        
        # 각 timekey에 대해 처리
        for timekey in self.timekey_starts.keys():
            start = self.timekey_starts[timekey]  # 현재 시간 키의 시작 인덱스
            end = self.timekey_ends[timekey]  # 현재 시간 키의 끝 인덱스
            tk_length = end - start  # 현재 시간 키의 데이터 길이
            
            # 데이터가 윈도우 크기보다 작으면 스킵
            if tk_length < self.window_size:
                continue
            
            # 벡터화된 window sliding
            # 생성 가능한 윈도우 개수 계산
            num_windows = (tk_length - self.window_size) // self.window_stride + 1
            
            # 각 윈도우에 대해 처리
            for w in range(num_windows):
                # 윈도우 시작과 끝 인덱스 계산
                window_start = start + w * self.window_stride
                window_end = window_start + self.window_size
                # 윈도우에 해당하는 인덱스 배열 생성
                window_indices = np.arange(window_start, window_end)
                
                # 참조 공정 정보 (직접 인덱싱)
                ref_global_idx = window_indices[ref_idx]  # 참조 인덱스의 전역 위치
                reference_oper_group = self.oper_groups[ref_global_idx]  # 참조 공정의 그룹
                reference_oper_id = self.oper_ids[ref_global_idx]  # 참조 공정의 ID
                
                # 그룹 마스크 생성 (벡터화)
                tk_indices = np.arange(start, end)  # 현재 시간의 모든 인덱스
                # 참조 그룹과 같은 그룹인지 확인하는 마스크
                group_mask = (self.oper_groups[start:end] == reference_oper_group)
                # 같은 그룹의 인덱스들
                group_indices_local = tk_indices[group_mask]
                
                # Window에 포함되지 않은 그룹 인덱스만 선택
                # setdiff1d: 첫 번째 배열에만 있고 두 번째 배열에는 없는 요소 반환
                group_indices = np.setdiff1d(group_indices_local, window_indices)
                
                # 시퀀스 정보 저장
                self.sequences.append({
                    'window_indices': window_indices,  # 윈도우 인덱스
                    'group_indices': group_indices,  # 그룹 인덱스
                    'reference_oper_id': reference_oper_id,  # 참조 공정 ID
                    'reference_oper_group': reference_oper_group,  # 참조 그룹
                    'timekey': timekey  # 시간 키
                })
    
    def __len__(self):
        # 데이터셋의 크기 (시퀀스 개수) 반환
        return len(self.sequences)
    
    def __getitem__(self, idx):
        """최적화된 데이터 로딩"""
        # 주어진 인덱스의 시퀀스 가져오기
        sequence = self.sequences[idx]
        window_indices = sequence['window_indices']  # 윈도우 인덱스
        group_indices = sequence['group_indices']  # 그룹 인덱스
        
        # 그룹 크기 제한
        group_size = min(len(group_indices), self.max_group_size)  # 최대 그룹 크기로 제한
        if group_size > 0:
            group_indices = group_indices[:group_size]  # 그룹 인덱스 슬라이싱
        
        # 사전 할당된 텐서 (torch.zeros가 np.full보다 빠름)
        # 연속형 데이터 텐서 (패딩 값으로 초기화)
        continuous_data = torch.full(
            (self.total_sequence_length, self.continuous_dim),
            self.padding_value, dtype=torch.float32
        )
        # 범주형 데이터 텐서 (0으로 초기화)
        categorical_data = torch.zeros(
            (self.total_sequence_length, self.categorical_dim),
            dtype=torch.float32
        )
        # 타겟 텐서 (패딩 값으로 초기화)
        targets = torch.full(
            (self.total_sequence_length,),
            self.padding_value, dtype=torch.float32
        )
        
        # Window 데이터 복사 (torch.from_numpy로 zero-copy view)
        # 윈도우 부분에 연속형 데이터 복사
        continuous_data[:self.window_size] = torch.from_numpy(self.continuous_array[window_indices])
        if self.categorical_dim > 0:  # 범주형 특성이 있는 경우
            # 윈도우 부분에 범주형 데이터 복사
            categorical_data[:self.window_size] = torch.from_numpy(self.categorical_array[window_indices])
        # 윈도우 부분에 타겟 데이터 복사
        targets[:self.window_size] = torch.from_numpy(self.target_array[window_indices])
        
        # Group 데이터 복사
        if group_size > 0:  # 그룹 데이터가 있는 경우
            # 그룹 부분에 연속형 데이터 복사
            continuous_data[self.window_size:self.window_size + group_size] = \
                torch.from_numpy(self.continuous_array[group_indices])
            if self.categorical_dim > 0:  # 범주형 특성이 있는 경우
                # 그룹 부분에 범주형 데이터 복사
                categorical_data[self.window_size:self.window_size + group_size] = \
                    torch.from_numpy(self.categorical_array[group_indices])
            # 그룹 부분에 타겟 데이터 복사
            targets[self.window_size:self.window_size + group_size] = \
                torch.from_numpy(self.target_array[group_indices])
        
        # 마스크와 position_ids (torch 텐서로 직접 생성)
        # 유효 데이터 마스크 (False로 초기화)
        masks = torch.zeros(self.total_sequence_length, dtype=torch.bool)
        # 실제 데이터가 있는 부분만 True로 설정
        masks[:self.window_size + group_size] = True
        
        # 위치 ID 텐서 (-1로 초기화)
        position_ids = torch.full((self.total_sequence_length,), -1, dtype=torch.long)
        # 윈도우 부분은 0으로 설정
        position_ids[:self.window_size] = 0
        if group_size > 0:  # 그룹 데이터가 있는 경우
            # 그룹 부분은 1로 설정
            position_ids[self.window_size:self.window_size + group_size] = 1
        
        # oper_ids 리스트 (필요한 경우만)
        # 윈도우 공정 ID 가져오기
        window_oper_ids = self.oper_ids[window_indices]
        # 전체 시퀀스 길이만큼 None으로 초기화
        oper_ids_list = [None] * self.total_sequence_length
        # 윈도우 공정 ID 채우기
        for i, oper_id in enumerate(window_oper_ids):
            oper_ids_list[i] = oper_id
        if group_size > 0:  # 그룹 데이터가 있는 경우
            # 그룹 공정 ID 가져오기
            group_oper_ids = self.oper_ids[group_indices]
            # 그룹 공정 ID 채우기
            for i, oper_id in enumerate(group_oper_ids):
                oper_ids_list[self.window_size + i] = oper_id
                
        # 최종 데이터 딕셔너리 반환
        return {
            'continuous_data': continuous_data,  # 연속형 특성
            'categorical_data': categorical_data,  # 범주형 특성
            'targets': targets,  # 타겟
            'masks': masks,  # 유효 데이터 마스크
            'position_ids': position_ids,  # 위치 구분자
            'sequence_lengths': self.window_size + group_size,  # 실제 시퀀스 길이
            'timekey': sequence['timekey'],  # 시간 키
            'oper_ids_list': oper_ids_list,  # 공정 ID 리스트
            'reference_oper_id': sequence['reference_oper_id'],  # 참조 공정 ID
            'reference_oper_group': sequence['reference_oper_group'],  # 참조 그룹
            'group_position': self.group_position  # 그룹 결정 위치
        }


def custom_collate_fn(batch):
    """최적화된 배치 함수"""
    # 스택 연산 최적화 (리스트 컴프리헨션 대신 직접 스택)
    batch_size = len(batch)  # 배치 크기
    
    # 텐서들을 한 번에 스택
    # 각 배치 아이템의 continuous_data를 스택하여 배치 텐서 생성
    continuous_data = torch.stack([item['continuous_data'] for item in batch])
    # 각 배치 아이템의 categorical_data를 스택
    categorical_data = torch.stack([item['categorical_data'] for item in batch])
    # 각 배치 아이템의 targets를 스택
    targets = torch.stack([item['targets'] for item in batch])
    # 각 배치 아이템의 masks를 스택
    masks = torch.stack([item['masks'] for item in batch])
    # 각 배치 아이템의 position_ids를 스택
    position_ids = torch.stack([item['position_ids'] for item in batch])
    
    # 리스트 데이터는 그대로
    return {
        'continuous_data': continuous_data,  # 배치 연속형 데이터
        'categorical_data': categorical_data,  # 배치 범주형 데이터
        'targets': targets,  # 배치 타겟
        'masks': masks,  # 배치 마스크
        'position_ids': position_ids,  # 배치 위치 ID
        'sequence_lengths': [item['sequence_lengths'] for item in batch],  # 시퀀스 길이 리스트
        'timekeys': [item['timekey'] for item in batch],  # 시간 키 리스트
        'oper_ids_list': [item['oper_ids_list'] for item in batch],  # 공정 ID 리스트들
        'reference_oper_ids': [item['reference_oper_id'] for item in batch],  # 참조 공정 ID 리스트
        'reference_oper_groups': [item['reference_oper_group'] for item in batch],  # 참조 그룹 리스트
        'group_positions': [item['group_position'] for item in batch]  # 그룹 위치 리스트
    }


def split_data_by_days(
    df: pd.DataFrame,  # 입력 데이터프레임
    train_ratio: float = 0.8,  # 학습 데이터 비율
    val_ratio: float = 0.1,  # 검증 데이터 비율
    test_ratio: float = 0.1  # 테스트 데이터 비율
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """날짜 기준으로 데이터를 분할"""
    
    # timekey_hr에서 날짜(day) 추출 (시간 정보 제거)
    # 예: 20240115 -> 202401 (년월 추출)
    df['date'] = (df['timekey_hr'].astype(int) // 100).astype(int)
    
    # 고유한 날짜들을 시간순으로 정렬
    unique_dates = sorted(df['date'].unique())
    total_days = len(unique_dates)  # 전체 날짜 수
    
    # 날짜 기준으로 분할 인덱스 계산
    train_days = int(total_days * train_ratio)  # 학습용 날짜 수
    val_days = int(total_days * val_ratio)  # 검증용 날짜 수
    
    # 각 분할에 해당하는 날짜 범위 설정
    train_dates = unique_dates[:train_days]  # 처음부터 train_days까지
    val_dates = unique_dates[train_days:train_days + val_days]  # train 다음부터 val_days개
    test_dates = unique_dates[train_days + val_days:]  # 나머지 모든 날짜
    
    # 각 분할에 해당하는 데이터 추출
    train_df = df[df['date'].isin(train_dates)].copy()  # 학습 데이터
    val_df = df[df['date'].isin(val_dates)].copy()  # 검증 데이터
    test_df = df[df['date'].isin(test_dates)].copy()  # 테스트 데이터
    
    # 분할 결과 정보 로깅
    logger.info(f"날짜 기준 데이터 분할 완료:")
    logger.info(f"  - 총 날짜 수: {total_days}일")
    logger.info(f"  - Train: {len(train_dates)}일 ({len(train_df):,}행)")
    logger.info(f"  - Validation: {len(val_dates)}일 ({len(val_df):,}행)")  
    logger.info(f"  - Test: {len(test_dates)}일 ({len(test_df):,}행)")
    logger.info(f"  - Train 날짜 범위: {min(train_dates)} ~ {max(train_dates)}")
    logger.info(f"  - Val 날짜 범위: {min(val_dates)} ~ {max(val_dates)}")
    logger.info(f"  - Test 날짜 범위: {min(test_dates)} ~ {max(test_dates)}")
    
    # 학습, 검증, 테스트 데이터프레임 반환
    return train_df, val_df, test_df


In [5]:
def create_dataloaders(config: Dict) -> Tuple[DataLoader, DataLoader, DataLoader]:

    # 데이터 로드 및 전처리
    data_path = config["file_path"]  # 설정에서 파일 경로 가져오기
    # nrows는 테스트용이므로 꼭 제거 ! (주의사항 메모)
    # Excel 파일의 모든 시트를 딕셔너리로 읽기, 두 번째 행을 헤더로 사용
    excel = pd.read_excel(data_path, sheet_name=None, header=1)
    sheet_names = config["sheet_names"]  # 사용할 시트 이름 목록 가져오기

    # 지정된 시트들의 데이터를 하나의 데이터프레임으로 결합
    total_df = pd.concat([excel[sheet_name] for sheet_name in sheet_names])
    
    # 각 그룹(oper_group)별로 속한 공정(oper_id)들의 고유 값을 딕셔너리로 생성
    group_to_opers = total_df.groupby('oper_group')['oper_id'].unique().to_dict()
    # 전체 고유 그룹 수 계산
    unique_groups_num = len(total_df['oper_group'].unique().tolist())

    # 기본 전처리
    # "Unnamed: 0" 컬럼이 있으면 제거 (Excel 읽기 시 생성되는 인덱스 컬럼)
    if "Unnamed: 0" in total_df.columns:
        total_df.drop(columns="Unnamed: 0", inplace=True)

    # y값 결측치 제거
    # 타겟 컬럼에 결측치가 없는 행만 선택하여 복사본 생성
    df = total_df[~total_df[config["target_column"]].isna()].copy()

    # 불필요한 컬럼 제거
    # 설정에서 추가로 제거할 컬럼 목록 가져오기 (없으면 빈 리스트)
    drop_columns = config.get("additional_drop_columns", [])
    if drop_columns:  # 제거할 컬럼이 있으면
        # 실제로 데이터프레임에 존재하는 컬럼만 필터링
        existing_drops = [col for col in drop_columns if col in df.columns]
        if existing_drops:  # 존재하는 컬럼이 있으면
            df = df.drop(columns=existing_drops)  # 해당 컬럼들 제거

    # 인덱스 재설정 (0부터 시작하는 연속 인덱스로 변경)
    df.reset_index(drop=True, inplace=True)

    # ========================================
    # 데이터 전처리 (정규화 전): NaN, Inf 제거
    # ========================================
    
    continuous_columns = config["continuous_columns"]
    categorical_columns = config["categorical_columns"]
    target_column = config["target_column"]
    
    logger.info("데이터 전처리 시작...")
    
    # NaN 체크 및 처리
    nan_count = df[continuous_columns].isna().sum().sum()
    if nan_count > 0:
        logger.warning(f"연속형 변수에 {nan_count}개의 NaN 발견! 0으로 대체합니다.")
        df[continuous_columns] = df[continuous_columns].fillna(0)
    
    # Inf 체크 및 제거
    inf_mask = np.isinf(df[continuous_columns].values)
    if inf_mask.any():
        logger.warning(f"Inf 값 발견! 제거합니다.")
        df[continuous_columns] = df[continuous_columns].replace([np.inf, -np.inf], np.nan)
        df[continuous_columns] = df[continuous_columns].fillna(0)
        
    # ========================================
    # 범주형 변수 처리 (분할 전에 전체 데이터로)
    # ========================================
    
    # 범주형 처리기는 전체 데이터로 학습 (모든 카테고리를 알아야 함)
    categorical_processor = CategoricalProcessor(
        embedding_dim=config.get("embedding_dim", 8)
    )
    categorical_processor.fit(df, categorical_columns)
    df = categorical_processor.transform(df)
    
    # ========================================
    # 데이터 분할
    # ========================================
    
    # 데이터 분할 (8:1:1)
    # 날짜 기준으로 시계열 데이터를 train, validation, test로 분할
    train_df, val_df, test_df = split_data_by_days(
        df,
        train_ratio=config.get("train_ratio", 0.8),  # 학습 데이터 비율 (기본값: 0.8)
        val_ratio=config.get("val_ratio", 0.1),      # 검증 데이터 비율 (기본값: 0.1)
        test_ratio=config.get("test_ratio", 0.1)      # 테스트 데이터 비율 (기본값: 0.1)
    )
    
    # ========================================
    # 연속형 변수 정규화
    # ========================================
    
    logger.info("연속형 변수 정규화")
    
    # 1. 극단값 클리핑 (각 데이터셋 별도 처리)
    if config.get("clip_outliers", True):
        logger.info("극단값 클리핑 적용...")
        
        # Train 데이터의 분위수로 클리핑 범위 결정
        clip_values = {}
        for col in continuous_columns:
            q01 = train_df[col].quantile(0.01)
            q99 = train_df[col].quantile(0.99)
            clip_values[col] = (q01, q99)
            
            # 모든 데이터셋에 동일한 클리핑 적용
            train_df[col] = train_df[col].clip(q01, q99)
            val_df[col] = val_df[col].clip(q01, q99)
            test_df[col] = test_df[col].clip(q01, q99)
            
            logger.debug(f"  {col}: [{q01:.4f}, {q99:.4f}]로 클리핑")
    
    # 2. Scaler 생성 및 학습 (train 데이터만 사용!)
    scaler_type = config.get("scaler_type", "robust")
    
    if scaler_type == "robust":
        continuous_scaler = RobustScaler()
        logger.info("RobustScaler 사용 (이상치에 강건, 중앙값과 IQR 기반)")
    elif scaler_type == "standard":
        continuous_scaler = StandardScaler()
        logger.info("StandardScaler 사용 (평균 0, 표준편차 1)")
    elif scaler_type == "minmax":
        # MinMaxScaler의 범위를 설정 가능하게
        feature_range = config.get("minmax_range", (0, 1))
        continuous_scaler = MinMaxScaler(feature_range=feature_range)
        logger.info(f"MinMaxScaler 사용 (범위: {feature_range[0]}~{feature_range[1]})")
    else:
        logger.warning(f"알 수 없는 scaler_type: {scaler_type}. RobustScaler 사용")
        continuous_scaler = RobustScaler()
    
    # Train 데이터로만 fit!
    continuous_scaler.fit(train_df[continuous_columns])
    
    # 모든 데이터에 transform 적용
    train_df[continuous_columns] = continuous_scaler.transform(train_df[continuous_columns])
    val_df[continuous_columns] = continuous_scaler.transform(val_df[continuous_columns])
    test_df[continuous_columns] = continuous_scaler.transform(test_df[continuous_columns])
    
    logger.info("Normalize done!")
    
    # 3. 타겟 변수 정규화 (옵션 - 일반적으로 권장하지 않음)
    target_scaler = None
    if config.get("normalize_target", False):
        # 타겟에도 동일한 스케일러 타입 적용 가능
        target_scaler_type = config.get("target_scaler_type", "robust")
        
        if target_scaler_type == "robust":
            target_scaler = RobustScaler()
        elif target_scaler_type == "standard":
            target_scaler = StandardScaler()
        elif target_scaler_type == "minmax":
            target_range = config.get("target_minmax_range", (0, 1))
            target_scaler = MinMaxScaler(feature_range=target_range)
        else:
            target_scaler = RobustScaler()
        
        # Train 데이터로만 fit!
        target_scaler.fit(train_df[[target_column]])
        
        # 모든 데이터에 transform 적용
        train_df[target_column] = target_scaler.transform(train_df[[target_column]])
        val_df[target_column] = target_scaler.transform(val_df[[target_column]])
        test_df[target_column] = target_scaler.transform(test_df[[target_column]])
        
        logger.info(f"타겟 변수 정규화 완료 ({target_scaler_type})")
    else:
        logger.info(f"타겟 변수 범위:")
        logger.info(f"  Train: [{train_df[target_column].min():.4f}, {train_df[target_column].max():.4f}]")
        logger.info(f"  Val: [{val_df[target_column].min():.4f}, {val_df[target_column].max():.4f}]")
        logger.info(f"  Test: [{test_df[target_column].min():.4f}, {test_df[target_column].max():.4f}]")
    
    
    # 데이터셋 생성 - group_position 파라미터 추가
    # 학습용 데이터셋 생성
    train_dataset = GroupedOperDataset(
        df=train_df,  # 학습용 데이터프레임
        categorical_columns=config["categorical_columns"],  # 범주형 컬럼 목록
        continuous_columns=config["continuous_columns"],    # 연속형 컬럼 목록
        target_column=config["target_column"],              # 타겟 컬럼명
        group_map=group_to_opers,                          # 그룹-공정 매핑 정보
        window_size=config.get("window_size", 10),         # 윈도우 크기 (기본값: 10)
        window_stride=config.get("window_stride", 1),      # 윈도우 이동 간격 (기본값: 1)
        padding_value=config.get("padding_value", 0.0),    # 패딩 값 (기본값: 0.0)
        group_position=config.get("group_position", "middle")  # 그룹 결정 기준 위치 (기본값: "middle")
    )

    # 검증용 데이터셋 생성 (학습 데이터셋과 동일한 설정 사용)
    val_dataset = GroupedOperDataset(
        df=val_df,  # 검증용 데이터프레임
        categorical_columns=config["categorical_columns"],
        continuous_columns=config["continuous_columns"], 
        target_column=config["target_column"],
        group_map=group_to_opers,
        window_size=config.get("window_size", 10),
        window_stride=config.get("window_stride", 1),
        padding_value=config.get("padding_value", 0.0),
        group_position=config.get("group_position", "middle")  # 추가
    )

    # 테스트용 데이터셋 생성 (학습 데이터셋과 동일한 설정 사용)
    test_dataset = GroupedOperDataset(
        df=test_df,  # 테스트용 데이터프레임
        categorical_columns=config["categorical_columns"],
        continuous_columns=config["continuous_columns"],
        target_column=config["target_column"],
        group_map=group_to_opers,
        window_size=config.get("window_size", 10),
        window_stride=config.get("window_stride", 1),
        padding_value=config.get("padding_value", 0.0),
        group_position=config.get("group_position", "middle")  # 추가
    )

    # 데이터로더 생성
    batch_size = config.get("batch_size", 32)  # 배치 크기 (기본값: 32)
    num_workers = config.get("num_workers", 4)  # 데이터 로딩 워커 수 (기본값: 4)
    # 그룹 참조 인덱스 계산 (그룹을 결정할 윈도우 내 위치)
    ref_idx = train_dataset._get_group_reference_index(train_dataset.window_size)

    # 학습용 데이터로더 생성
    train_loader = DataLoader(
        train_dataset,              # 학습 데이터셋
        batch_size=batch_size,       # 배치 크기
        shuffle=True,                # 학습 시 데이터 섞기 활성화
        num_workers=num_workers,     # 병렬 데이터 로딩 워커 수
        pin_memory=True,             # GPU 메모리로 빠른 전송을 위한 고정 메모리 사용
        collate_fn=custom_collate_fn # 배치 데이터 결합 함수
    )

    # 검증용 데이터로더 생성
    val_loader = DataLoader(
        val_dataset,                 # 검증 데이터셋
        batch_size=batch_size,       # 배치 크기
        shuffle=False,               # 검증 시 데이터 섞기 비활성화 (재현성)
        num_workers=num_workers,     # 병렬 데이터 로딩 워커 수
        pin_memory=True,             # GPU 메모리로 빠른 전송을 위한 고정 메모리 사용
        collate_fn=custom_collate_fn # 배치 데이터 결합 함수
    )

    # 테스트용 데이터로더 생성
    test_loader = DataLoader(
        test_dataset,                # 테스트 데이터셋
        batch_size=batch_size,       # 배치 크기
        shuffle=False,               # 테스트 시 데이터 섞기 비활성화 (재현성)
        num_workers=num_workers,     # 병렬 데이터 로딩 워커 수
        pin_memory=True,             # GPU 메모리로 빠른 전송을 위한 고정 메모리 사용
        collate_fn=custom_collate_fn # 배치 데이터 결합 함수
    )
    
    # 데이터로더 생성 정보 로깅
    logger.info(f"데이터로더 생성 완료:")
    logger.info(f"  - 훈련 샘플: {len(train_dataset)}")     # 학습 데이터셋 크기
    logger.info(f"  - 검증 샘플: {len(val_dataset)}")       # 검증 데이터셋 크기
    logger.info(f"  - 테스트 샘플: {len(test_dataset)}")    # 테스트 데이터셋 크기
    logger.info(f"  - 배치 크기: {batch_size}")             # 배치 크기
    logger.info(f"  - 그룹 기준 위치: {config.get('group_position', 'first')}")  # 그룹 결정 위치

    # 데이터로더 3개와 추가 정보 반환
    return train_loader, val_loader, test_loader, categorical_processor, ref_idx, unique_groups_num
    # train_loader: 학습용 데이터로더
    # val_loader: 검증용 데이터로더
    # test_loader: 테스트용 데이터로더
    # categorical_processor: 범주형 데이터 처리기 (추론 시 동일한 인코딩 적용)
    # ref_idx: 그룹 참조 인덱스 (모델에서 사용)
    # unique_groups_num: 고유 그룹 수 (모델 설계 참고용)

## OperTransformer모델

In [6]:
"""Encoding Modules"""

class PositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEmbedding, self).__init__()
        # Compute the positional encodings once in log space.
        pe = torch.zeros(max_len, d_model).float()
        pe.require_grad = False

        position = torch.arange(0, max_len).float().unsqueeze(1)
        div_term = (torch.arange(0, d_model, 2).float()
                    * -(math.log(10000.0) / d_model)).exp()

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return self.pe[:, :x.size(1)]


class CategoricalEmbedding(nn.Module):
    """범주형 변수를 위한 임베딩 모듈"""
    def __init__(self, vocab_sizes: List[int], embedding_dim: int):
        super(CategoricalEmbedding, self).__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(vocab_size, embedding_dim) 
            for vocab_size in vocab_sizes
        ])
        
    def forward(self, categorical_inputs):
        """
        Args:
            categorical_inputs: [batch_size, seq_len, num_categorical_features]
        Returns:
            embedded: [batch_size, seq_len, num_categorical_features * embedding_dim]
        """
        embedded_features = []
        for i, embedding_layer in enumerate(self.embeddings):
            # 각 범주형 변수에 대해 임베딩 수행
            cat_input = categorical_inputs[:, :, i].long()
            embedded = embedding_layer(cat_input)
            embedded_features.append(embedded)
        
        # 모든 임베딩을 concatenate
        return torch.cat(embedded_features, dim=-1)


class DataEmbeddingWithCategorical(nn.Module):
    """범주형 및 연속형 변수를 모두 처리하는 임베딩 모듈 (특별 토큰 포함)"""
    def __init__(self, 
                 continuous_dim: int,
                 vocab_sizes: List[int],
                 d_model: int,
                 window_size: int = 10,
                 embedding_dim: int = 8,
                 dropout: float = 0.1,
                 use_special_tokens: bool = False,
                 num_groups: int = None):
        super(DataEmbeddingWithCategorical, self).__init__()
        
        self.window_size = window_size
        self.use_special_tokens = use_special_tokens
        self.d_model = d_model
        
        # 범주형 변수 임베딩
        self.categorical_embedding = CategoricalEmbedding(vocab_sizes, embedding_dim)
        
        # 범주형 임베딩과 연속형 변수를 결합한 차원
        total_input_dim = continuous_dim + len(vocab_sizes) * embedding_dim
        
        # 입력을 d_model 차원으로 투영
        self.input_projection = nn.Linear(total_input_dim, d_model)
        
        # 특별 토큰 (learnable parameters)
        if use_special_tokens:
            self.seq_token = nn.Parameter(torch.randn(1, 1, d_model))
            
            if num_groups is not None:
                self.group_tokens = nn.Parameter(torch.randn(num_groups, 1, d_model))
            else:
                self.group_token_embedding = nn.Embedding(1000, d_model)
                
        # Positional 임베딩 (Window 크기에 맞춤)
        self.position_embedding = PositionalEmbedding(d_model, max_len=window_size)
        
        self.dropout = nn.Dropout(p=dropout)
        self.norm = nn.LayerNorm(d_model)
            
    def forward(self, continuous_x, categorical_x, position_ids=None, reference_groups=None):
        batch_size, seq_len = continuous_x.shape[:2]
        
        # 범주형 변수 임베딩
        cat_embedded = self.categorical_embedding(categorical_x)
        
        # 연속형과 범주형 결합
        combined = torch.cat([continuous_x, cat_embedded], dim=-1)
        
        # d_model 차원으로 투영
        x = self.input_projection(combined)
        
        # Window 부분에만 positional embedding 적용
        if position_ids is not None:
            for b in range(batch_size):
                window_mask = (position_ids[b] == 0)
                window_indices = window_mask.nonzero(as_tuple=False).squeeze(-1)
                
                if len(window_indices) > 0:
                    window_length = len(window_indices)
                    pos_emb = self.position_embedding.pe[:, :window_length, :].squeeze(0)
                    x[b, window_indices] = x[b, window_indices] + pos_emb
        else:
            if seq_len <= self.window_size:
                x = x + self.position_embedding(x)
            else:
                pos_encoding = self.position_embedding.pe[:, :self.window_size, :]
                x[:, :self.window_size, :] = x[:, :self.window_size, :] + pos_encoding
                
        # 특별 토큰 추가
        if self.use_special_tokens and position_ids is not None and reference_groups is not None:
            new_x = []
            new_masks = []
            
            for b in range(batch_size):
                # 각 부분 추출
                padding_mask = (position_ids[b] == -1)
                window_mask = (position_ids[b] == 0)
                group_mask = (position_ids[b] == 1)
                
                window_data = x[b, window_mask]
                group_data = x[b, group_mask]
                padding_data = x[b, padding_mask]
                
                # 새로운 시퀀스 구성
                seq_parts = []
                mask_parts = []
                
                # 1. [SEQ] 토큰
                seq_token = self.seq_token.view(1, -1)
                seq_parts.append(seq_token)
                mask_parts.append(torch.ones(1, device=x.device, dtype=torch.bool))
                
                # 2. Window 데이터
                if window_data.numel() > 0:
                    if window_data.dim() == 1:
                        window_data = window_data.unsqueeze(0)
                    seq_parts.append(window_data)
                    mask_parts.append(torch.ones(window_data.shape[0], device=x.device, dtype=torch.bool))
                
                # 3. [GROUP] 토큰
                group_id = reference_groups[b].item() if hasattr(reference_groups[b], 'item') else reference_groups[b]
                if hasattr(self, 'group_tokens'):
                    group_token = self.group_tokens[group_id].view(1, -1)
                else:
                    group_token = self.group_token_embedding(
                        torch.tensor([group_id], device=x.device)
                    ).view(1, -1)
                seq_parts.append(group_token)
                mask_parts.append(torch.ones(1, device=x.device, dtype=torch.bool))
                
                # 4. Group 데이터
                if group_data.numel() > 0:
                    if group_data.dim() == 1:
                        group_data = group_data.unsqueeze(0)
                    seq_parts.append(group_data)
                    mask_parts.append(torch.ones(group_data.shape[0], device=x.device, dtype=torch.bool))
                
                # 5. 패딩 데이터 (마스크는 False)
                if padding_data.numel() > 0:
                    if padding_data.dim() == 1:
                        padding_data = padding_data.unsqueeze(0)
                    seq_parts.append(padding_data)
                    mask_parts.append(torch.zeros(padding_data.shape[0], device=x.device, dtype=torch.bool))  # 패딩은 False
                    
                # 결합
                new_seq = torch.cat(seq_parts, dim=0)
                new_mask = torch.cat(mask_parts, dim=0)
                
                new_x.append(new_seq)
                new_masks.append(new_mask)
            
            # 모든 시퀀스가 같은 길이이므로 추가 패딩 불필요
            # 바로 스택 가능
            x = torch.stack(new_x)  # [batch_size, seq_len + 2, d_model]
            masks = torch.stack(new_masks)  # [batch_size, seq_len + 2]
            
            # Normalization과 Dropout
            x = self.norm(self.dropout(x))
            return x, masks
        
        # 특별 토큰 없이
        x = self.norm(self.dropout(x))
        masks = (position_ids != -1) if position_ids is not None else torch.ones((batch_size, seq_len), dtype=torch.bool, device=x.device)
        return x, masks

"""Encoder/Decoder Modules"""

class ConvLayer(nn.Module):
    def __init__(self, c_in):
        super(ConvLayer, self).__init__()
        self.downConv = nn.Conv1d(in_channels=c_in,
                                  out_channels=c_in,
                                  kernel_size=3,
                                  padding=2,
                                  padding_mode='circular')
        self.norm = nn.BatchNorm1d(c_in)
        self.activation = nn.ELU()
        self.maxPool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)

    def forward(self, x):
        x = self.downConv(x.permute(0, 2, 1))
        x = self.norm(x)
        x = self.activation(x)
        x = self.maxPool(x)
        x = x.transpose(1, 2)
        return x


class EncoderLayer(nn.Module):
    def __init__(self, attention, d_model, d_ff=None, dropout=0.1, activation="relu"):
        super(EncoderLayer, self).__init__()
        d_ff = d_ff or 4 * d_model
        self.attention = attention
        self.conv1 = nn.Conv1d(in_channels=d_model, out_channels=d_ff, kernel_size=1)
        self.conv2 = nn.Conv1d(in_channels=d_ff, out_channels=d_model, kernel_size=1)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.activation = F.relu if activation == "relu" else F.gelu

    def forward(self, x, attn_mask=None, tau=None, delta=None):
        new_x, attn = self.attention(
            x, x, x,
            attn_mask=attn_mask,
            tau=tau, delta=delta
        )
        x = x + self.dropout(new_x)

        y = x = self.norm1(x)
        y = self.dropout(self.activation(self.conv1(y.transpose(-1, 1))))
        y = self.dropout(self.conv2(y).transpose(-1, 1))

        return self.norm2(x + y), attn


class Encoder(nn.Module):
    def __init__(self, attn_layers, conv_layers=None, norm_layer=None):
        super(Encoder, self).__init__()
        self.attn_layers = nn.ModuleList(attn_layers)
        self.conv_layers = nn.ModuleList(conv_layers) if conv_layers is not None else None
        self.norm = norm_layer

    def forward(self, x, attn_mask=None, tau=None, delta=None):
        # x [B, L, D]
        attns = []
        if self.conv_layers is not None:
            for i, (attn_layer, conv_layer) in enumerate(zip(self.attn_layers, self.conv_layers)):
                delta = delta if i == 0 else None
                x, attn = attn_layer(x, attn_mask=attn_mask, tau=tau, delta=delta)
                x = conv_layer(x)
                attns.append(attn)
            x, attn = self.attn_layers[-1](x, tau=tau, delta=None)
            attns.append(attn)
        else:
            for attn_layer in self.attn_layers:
                x, attn = attn_layer(x, attn_mask=attn_mask, tau=tau, delta=delta)
                attns.append(attn)

        if self.norm is not None:
            x = self.norm(x)

        return x, attns


class DecoderLayer(nn.Module):
    def __init__(self, self_attention, cross_attention, d_model, d_ff=None,
                 dropout=0.1, activation="relu"):
        super(DecoderLayer, self).__init__()
        d_ff = d_ff or 4 * d_model
        self.self_attention = self_attention
        self.cross_attention = cross_attention
        self.conv1 = nn.Conv1d(in_channels=d_model, out_channels=d_ff, kernel_size=1)
        self.conv2 = nn.Conv1d(in_channels=d_ff, out_channels=d_model, kernel_size=1)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.activation = F.relu if activation == "relu" else F.gelu

    def forward(self, x, cross, x_mask=None, cross_mask=None, tau=None, delta=None):
        x = x + self.dropout(self.self_attention(
            x, x, x,
            attn_mask=x_mask,
            tau=tau, delta=None
        )[0])
        x = self.norm1(x)

        x = x + self.dropout(self.cross_attention(
            x, cross, cross,
            attn_mask=cross_mask,
            tau=tau, delta=delta
        )[0])

        y = x = self.norm2(x)
        y = self.dropout(self.activation(self.conv1(y.transpose(-1, 1))))
        y = self.dropout(self.conv2(y).transpose(-1, 1))

        return self.norm3(x + y)


class Decoder(nn.Module):
    def __init__(self, layers, norm_layer=None, projection=None):
        super(Decoder, self).__init__()
        self.layers = nn.ModuleList(layers)
        self.norm = norm_layer
        self.projection = projection

    def forward(self, x, cross, x_mask=None, cross_mask=None, tau=None, delta=None):
        for layer in self.layers:
            x = layer(x, cross, x_mask=x_mask, cross_mask=cross_mask, tau=tau, delta=delta)

        if self.norm is not None:
            x = self.norm(x)

        if self.projection is not None:
            x = self.projection(x)
        return x

"""Attention Modules"""

class AttentionLayer(nn.Module):
    def __init__(self, attention, d_model, n_heads, d_keys=None,
                 d_values=None):
        super(AttentionLayer, self).__init__()

        d_keys = d_keys or (d_model // n_heads)
        d_values = d_values or (d_model // n_heads)

        self.inner_attention = attention
        self.query_projection = nn.Linear(d_model, d_keys * n_heads)
        self.key_projection = nn.Linear(d_model, d_keys * n_heads)
        self.value_projection = nn.Linear(d_model, d_values * n_heads)
        self.out_projection = nn.Linear(d_values * n_heads, d_model)
        self.n_heads = n_heads

    def forward(self, queries, keys, values, attn_mask, tau=None, delta=None):
        B, L, _ = queries.shape
        _, S, _ = keys.shape
        H = self.n_heads

        queries = self.query_projection(queries).view(B, L, H, -1)
        keys = self.key_projection(keys).view(B, S, H, -1)
        values = self.value_projection(values).view(B, S, H, -1)

        out, attn = self.inner_attention(
            queries,
            keys,
            values,
            attn_mask,
            tau=tau,
            delta=delta
        )
        out = out.view(B, L, -1)

        return self.out_projection(out), attn

class TriangularCausalMask():
    def __init__(self, B, L, device="cpu"):
        mask_shape = [B, 1, L, L]
        with torch.no_grad():
            self._mask = torch.triu(torch.ones(mask_shape, dtype=torch.bool), diagonal=1).to(device)

    @property
    def mask(self):
        return self._mask
    
class FullAttention(nn.Module):
    def __init__(self, mask_flag=True, factor=5, scale=None, attention_dropout=0.1, output_attention=False):
        super(FullAttention, self).__init__()
        self.scale = scale
        self.mask_flag = mask_flag
        self.output_attention = output_attention
        self.dropout = nn.Dropout(attention_dropout)

    def forward(self, queries, keys, values, attn_mask, tau=None, delta=None):
        B, L, H, E = queries.shape
        _, S, _, D = values.shape
        scale = self.scale or 1. / math.sqrt(E)

        scores = torch.einsum("blhe,bshe->bhls", queries, keys)

        if self.mask_flag:
            if attn_mask is None:
                attn_mask = TriangularCausalMask(B, L, device=queries.device)

            scores.masked_fill_(attn_mask.mask, -np.inf)

        A = self.dropout(torch.softmax(scale * scores, dim=-1))
        V = torch.einsum("bhls,bshd->blhd", A, values)

        if self.output_attention:
            return V.contiguous(), A
        else:
            return V.contiguous(), None


"""Model"""

class VanillaTransformer(nn.Module):
    """사용자 데이터 구조에 맞춘 수정된 Vanilla Transformer"""
    
    def __init__(self, configs):
        super(VanillaTransformer, self).__init__()
        
        self.ref_idx = configs.ref_idx  # 그룹 참조 인덱스 (window 내 위치)
        self.output_type = configs.output_type  # 'sequence' or 'target'
        self.use_special_tokens = configs.use_special_tokens
        self.window_size = configs.window_size
        
        # 데이터 임베딩 (범주형 + 연속형)
        self.data_embedding = DataEmbeddingWithCategorical(
            continuous_dim=configs.continuous_dim,
            vocab_sizes=configs.vocab_sizes,
            d_model=configs.d_model,
            embedding_dim=configs.embedding_dim,
            dropout=configs.dropout,
            window_size=configs.window_size,
            use_special_tokens=configs.use_special_tokens,
            num_groups=configs.num_groups
        )
        
        # Encoder
        self.encoder = Encoder(
            [
                EncoderLayer(
                    AttentionLayer(
                        FullAttention(
                            mask_flag=False,
                            factor=configs.factor,
                            attention_dropout=configs.dropout,
                            output_attention=False
                        ),
                        configs.d_model,
                        configs.n_heads
                    ),
                    configs.d_model,
                    configs.d_ff,
                    dropout=configs.dropout,
                    activation=configs.activation
                ) for _ in range(configs.e_layers)
            ],
            norm_layer=nn.LayerNorm(configs.d_model)
        )
        
        # Prediction head
        self.prediction_head = nn.Sequential(
            nn.Linear(configs.d_model, configs.d_ff),
            nn.ReLU(),
            nn.Dropout(configs.dropout),
            nn.Linear(configs.d_ff, 1)  # 단일 값 예측
        )
        
    def forward(self, continuous_data, categorical_data, position_ids, reference_oper_groups, targets, masks=None, sequence_lengths=None):
        """
        학습 코드와 호환되는 forward 메서드
        
        Args:
            continuous_data: [batch_size, seq_len, continuous_dim]
            categorical_data: [batch_size, seq_len, num_categorical]
            position_ids: [batch_size, seq_len] - 0 for window, 1 for group, -1 for padding
            reference_oper_groups: [batch_size] - 참조 그룹 ID
            targets: [batch_size, seq_len] - 타겟 값
            masks: [batch_size, seq_len] - True는 유효한 위치
            sequence_lengths: 실제 시퀀스 길이
            
        Returns:
            predictions: [batch_size, seq_len] or [batch_size]
            targets: 그대로 반환
        """
        
        # 임베딩 (특별 토큰이 추가되면 시퀀스 길이가 변경됨)
        enc_out, new_masks = self.data_embedding(continuous_data, categorical_data, position_ids, reference_oper_groups)
        
        # Attention mask 처리 (False = 마스킹할 위치)
        attn_mask = None
        if new_masks is not None:
            batch_size, seq_len = new_masks.shape
            # attention에서는 True가 마스킹 위치이므로 반전
            attn_mask = ~new_masks  
            attn_mask = attn_mask.unsqueeze(1).unsqueeze(1)  # [B, 1, 1, L]
            attn_mask = attn_mask.expand(-1, 1, seq_len, -1)  # [B, 1, L, L]
        
        # Encoder
        enc_out, attns = self.encoder(enc_out, attn_mask=attn_mask)
        
        if self.use_special_tokens:
            # 각 부분 추출
            seq_token = enc_out[:, 0, :]  # [B, d_model] - [SEQ] 토큰
            window_repr = enc_out[:, 1:self.window_size + 1, :]  # [B, window_size, d_model]
            group_token = enc_out[:, self.window_size + 1, :]  # [B, d_model] - [GROUP] 토큰
            
            if self.output_type == 'sequence':
                # Window 부분 전체에 대해 예측
                predictions = self.prediction_head(window_repr).squeeze(-1)  # [B, window_size]
                # targets도 window 부분만
                targets_out = targets[:, :self.window_size]
                
            elif self.output_type == 'target':
                # ref_idx 위치만 사용
                ref_repr = window_repr[:, self.ref_idx, :]  # [B, d_model]
                predictions = self.prediction_head(ref_repr).squeeze(-1)  # [B]
                
                # targets도 ref_idx 위치만 추출
                targets_out = targets[:, self.ref_idx]  # [B]
        
        else:
            # 특별 토큰 없는 경우
            if self.output_type == 'sequence':
                # 전체 시퀀스에 대해 예측
                predictions = self.prediction_head(enc_out).squeeze(-1)  # [B, seq_len]
                targets_out = targets
                
            elif self.output_type == 'target':
                # ref_idx 위치만 사용
                ref_repr = enc_out[:, self.ref_idx, :]  # [B, d_model]
                predictions = self.prediction_head(ref_repr).squeeze(-1)  # [B]
                
                # targets도 ref_idx 위치만
                targets_out = targets[:, self.ref_idx]  # [B]
                
        return predictions, targets_out

### 실험 설정 (모델 생성/Loss 설정)

In [7]:
def create_model(config: Dict, vocab_sizes: List[int], continuous_dim: int, ref_idx:int, unique_groups_num: int):
    """설정에 따른 모델 생성"""
    
    # 모델 설정 딕셔너리 생성
    model_config = {
        # 데이터 차원 관련 설정
        "continuous_dim": continuous_dim,  # 연속형 변수의 차원 수
        "vocab_sizes": vocab_sizes,  # 각 범주형 변수의 어휘 크기 (임베딩 테이블 크기 결정)
        "output_type": config.get("output_type", 'sequence'),  # 출력 타입 ('sequence' 또는 'target', 기본값: 'sequence')
        "use_special_tokens": config.get("use_special_tokens", True),  # 특수 토큰 사용 여부 (기본값: True)
        
        # 모델 하이퍼파라미터
        "seq_len": config.get("seq_len", 100),  # 입력 시퀀스 길이 (기본값: 100)
        "pred_len": config.get("pred_len", 1),  # 예측 길이 (기본값: 1)
        "window_size": config.get("window_size", 10),  # 윈도우 크기 (기본값: 10)
        
        # Transformer 설정
        "d_model": config.get("d_model", 512),  # 모델의 히든 차원 (기본값: 512)
        "n_heads": config.get("n_heads", 8),  # 멀티헤드 어텐션의 헤드 수 (기본값: 8)
        "e_layers": config.get("e_layers", 3),  # Encoder 레이어 수 (기본값: 3)
        "d_ff": config.get("d_ff", 2048),  # Feed Forward 네트워크의 히든 차원 (기본값: 2048)
        
        # 임베딩 설정
        "embedding_dim": config.get("embedding_dim", 8),  # 범주형 변수 임베딩 차원 (기본값: 8)
        "ref_idx": ref_idx,  # 그룹 결정 참조 인덱스 (윈도우 내에서 어느 위치를 기준으로 할지)
        "num_groups": unique_groups_num,  # 전체 그룹 수 (그룹 임베딩 테이블 크기 결정)
        
        # 학습 설정
        "dropout": config.get("dropout", 0.1),  # 드롭아웃 비율 (기본값: 0.1)
        "activation": config.get("activation", "relu"),  # 활성화 함수 (기본값: "relu")
        "factor": config.get("factor", 5),  # ProbAttention의 factor 파라미터 (기본값: 5)
        
    }
    
    # 딕셔너리를 SimpleNamespace 객체로 변환 (점 표기법으로 접근 가능하게 함)
    # 예: model_config['d_model'] 대신 model_config.d_model로 접근 가능
    model_config = SimpleNamespace(**model_config)
    
    model = VanillaTransformer(model_config)
    
    logger.info(f"모델 생성 완료:")
    logger.info(f"  - 총 파라미터 수: {sum(p.numel() for p in model.parameters()):,}")
    logger.info(f"  - 학습 가능한 파라미터 수: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    
    return model

class MSELoss(nn.Module):    
    def __init__(self, padding_value: float = 0.0):
        super().__init__()
        self.padding_value = padding_value
    
    def forward(self, predictions, targets, masks):
        """
        Args:
            predictions: [batch_size, seq_len]
            targets: [batch_size, seq_len]  
            masks: [batch_size, seq_len] (True = 패딩)
        """
        
        return F.mse_loss(predictions, targets)


def compute_metrics(predictions, targets, masks, padding_value: float = 0.0, exclude_zeros=True):
    """
    패딩을 고려한 메트릭 계산
    
    Args:
        predictions: 예측값 텐서
        targets: 실제값 텐서  
        masks: 패딩 마스크 (True가 패딩)
        padding_value: 패딩 값
        exclude_zeros: True면 MAPE/SMAPE 계산 시 타겟 0 제외,
                      False면 0도 포함 (epsilon으로 처리)
    """
    
    # CPU로 변환 후 numpy 배열로 변환
    valid_predictions = predictions.detach().cpu().numpy()
    valid_targets = targets.detach().cpu().numpy()
    
    # 기본 메트릭 (모든 유효 데이터 포함)
    mse = np.mean((valid_predictions - valid_targets) ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(valid_predictions - valid_targets))
    
    if exclude_zeros:
        # 타겟이 0이 아닌 경우만 필터링
        non_zero_mask = valid_targets != 0
        non_zero_count = non_zero_mask.sum()
        
        if non_zero_count > 0:
            filtered_predictions = valid_predictions[non_zero_mask]
            filtered_targets = valid_targets[non_zero_mask]
            
            abs_errors = np.abs(filtered_predictions - filtered_targets)
            abs_targets = np.abs(filtered_targets)
            
            # MAPE
            mape = np.mean(abs_errors / abs_targets * 100)
            
            # SMAPE
            smape = 100 * np.mean(
                2 * abs_errors / (abs_targets + np.abs(filtered_predictions) + 1e-8)
            )
        else:
            mape = np.nan
            smape = np.nan
            
    else:
        # 0도 포함하여 계산 (epsilon 사용)
        epsilon = 1e-8
        abs_targets = np.abs(valid_targets)
        abs_errors = np.abs(valid_predictions - valid_targets)
        
        # MAPE - epsilon으로 0 처리
        safe_targets = np.maximum(abs_targets, epsilon)
        mape = np.mean(abs_errors / safe_targets * 100)
        
        # SMAPE - epsilon으로 0 처리
        denominator = (abs_targets + np.abs(valid_predictions)) / 2 + epsilon
        smape = 100 * np.mean(abs_errors / denominator)
        
        non_zero_count = (valid_targets != 0).sum()
    
    return {
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "mape": mape,
        "smape": smape,
        "valid_count": len(valid_predictions),
        "non_zero_count": non_zero_count,
        "zero_ratio": (len(valid_targets) - non_zero_count) / len(valid_targets) * 100,
        "exclude_zeros": exclude_zeros  # 어떤 방식으로 계산했는지 표시
    }

In [8]:
def train_epoch(model, dataloader, criterion, optimizer, device, epoch):
    """한 에폭 훈련"""
    model.train()
    total_loss = 0.0
    total_metrics = {"mse": 0.0, "rmse": 0.0, "mae": 0.0, "mape": 0.0, "valid_count": 0}
    
    pbar = tqdm(
        enumerate(dataloader), 
        total=len(dataloader),
        desc=f"Epoch {epoch} [Train]",
        leave=False
    )
    
    for batch_idx, batch in pbar:
        continuous_data = batch["continuous_data"].to(device)
        categorical_data = batch["categorical_data"].to(device)
        position_ids = batch['position_ids']
        reference_oper_groups = batch['reference_oper_groups']
        targets = batch["targets"].to(device)
        masks = batch["masks"].to(device)
        sequence_lengths = batch["sequence_lengths"]
        
        optimizer.zero_grad()
        
        # Forward pass
        predictions, targets = model(continuous_data, categorical_data, position_ids, 
                            reference_oper_groups, targets, masks, sequence_lengths)
        loss = criterion(predictions, targets, masks)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # 메트릭 계산
        with torch.no_grad():
            batch_metrics = compute_metrics(predictions, targets, masks)
        
        total_loss += loss.item()
        for key in ["mse", "rmse", "mae", "mape"]:
            total_metrics[key] += batch_metrics[key]
        total_metrics["valid_count"] += batch_metrics["valid_count"]
        
        # 진행바 업데이트
        pbar.set_postfix({
            "Loss": f"{loss.item():.4f}",
            "MAPE": f"{batch_metrics['mape']:.2f}%"
        })
    
    pbar.close()
    
    # 평균 계산
    avg_loss = total_loss / len(dataloader)
    for key in ["mse", "rmse", "mae", "mape"]:
        total_metrics[key] = total_metrics[key] / len(dataloader)
    
    return avg_loss, total_metrics


def validate_epoch(model, dataloader, criterion, device, epoch=None):
    """검증 에폭"""
    model.eval()
    total_loss = 0.0
    total_metrics = {"mse": 0.0, "rmse": 0.0, "mae": 0.0, "mape": 0.0, "valid_count": 0}
    
    desc = f"Epoch {epoch} [Val]" if epoch is not None else "Validation"
    pbar = tqdm(dataloader, desc=desc, leave=False)
    
    with torch.no_grad():
        for batch in pbar:
            continuous_data = batch["continuous_data"].to(device)
            categorical_data = batch["categorical_data"].to(device)
            position_ids = batch['position_ids']
            reference_oper_groups = batch['reference_oper_groups']
            targets = batch["targets"].to(device)
            masks = batch["masks"].to(device)
            sequence_lengths = batch["sequence_lengths"]
            
            predictions, targets = model(continuous_data, categorical_data, position_ids, 
                            reference_oper_groups, targets, masks, sequence_lengths)
            loss = criterion(predictions, targets, masks)
            
            batch_metrics = compute_metrics(predictions, targets, masks)
            
            total_loss += loss.item()
            for key in ["mse", "rmse", "mae", "mape"]:
                total_metrics[key] += batch_metrics[key]
            total_metrics["valid_count"] += batch_metrics["valid_count"]
            
            pbar.set_postfix({
                "Loss": f"{loss.item():.4f}",
                "MAPE": f"{batch_metrics['mape']:.2f}%"
            })
    
    pbar.close()
    
    avg_loss = total_loss / len(dataloader)
    for key in ["mse", "rmse", "mae", "mape"]:
        total_metrics[key] = total_metrics[key] / len(dataloader)
    
    return avg_loss, total_metrics

In [9]:
def train_model(model, train_loader, val_loader, training_config, device, save_path):
    """메인 훈련 루프"""
    
    # 학습 설정 파라미터 추출
    num_epochs = training_config.get("num_epochs", 20)  # 전체 에폭 수 (기본값: 100)
    learning_rate = training_config.get("learning_rate", 1e-3)  # 학습률 (기본값: 0.001)
    patience = training_config.get("patience", 5)  # 조기 종료 인내도 (기본값: 20)
    padding_value = training_config.get("padding_value", 0.0)  # 패딩 값 (기본값: 0.0)
    
    # 손실 함수 및 옵티마이저 설정
    criterion = MSELoss(padding_value)  # 패딩 값을 고려한 MSE 손실 함수
    # AdamW 옵티마이저 (가중치 감쇠 포함)
    optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)
    # 학습률 스케줄러 (검증 손실이 개선되지 않으면 학습률 감소)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=patience//2, verbose=True)
    
    # 모델을 지정된 디바이스로 이동 (GPU/CPU)
    model = model.to(device)
    
    # 최적 모델 추적 변수 초기화
    best_val_loss = float('inf')  # 최저 검증 손실 (초기값: 무한대)
    patience_counter = 0  # 조기 종료 카운터
    
    # 학습 시작 로그
    logger.info(f"훈련 시작: {num_epochs} 에폭, 학습률 {learning_rate}")
    
    # 에폭 진행바 생성 (시각적 피드백)
    epoch_pbar = tqdm(range(1, num_epochs + 1), desc="Training Progress")
    
    # 각 에폭 반복
    for epoch in epoch_pbar:
        # 학습 에폭 수행 (한 에폭 동안 전체 학습 데이터 학습)
        train_loss, train_metrics = train_epoch(
            model, train_loader, criterion, optimizer, device, epoch
        )
        # 검증 에폭 수행 (검증 데이터로 성능 평가)
        val_loss, val_metrics = validate_epoch(
            model, val_loader, criterion, device, epoch
        )
        
        # 학습률 스케줄러 업데이트 (검증 손실 기반)
        scheduler.step(val_loss)
        
        # 로그 출력 (에폭 결과 요약)
        logger.info(
            f"Epoch {epoch:3d}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}, "
            f'Train MAPE={train_metrics["mape"]:.2f}%, Val MAPE={val_metrics["mape"]:.2f}%'
        )
        
        # 진행바 업데이트 (현재 상태 표시)
        epoch_pbar.set_postfix({
            "T_Loss": f"{train_loss:.4f}",  # 학습 손실
            "V_Loss": f"{val_loss:.4f}",  # 검증 손실
            "V_MAPE": f'{val_metrics["mape"]:.2f}%',  # 검증 MAPE
            "Best": f"{best_val_loss:.4f}",  # 최저 검증 손실
            "Patience": f"{patience_counter}/{patience}"  # 조기 종료 카운터
        })
        
        # 최고 모델 저장 로직
        if val_loss < best_val_loss:  # 검증 손실이 개선된 경우
            best_val_loss = val_loss  # 최저 손실 업데이트
            patience_counter = 0  # 카운터 리셋
            
            # 체크포인트 저장 (모델 상태, 옵티마이저 상태, 메트릭 등)
            torch.save({
                'epoch': epoch,  # 현재 에폭
                'model_state_dict': model.state_dict(),  # 모델 가중치
                'optimizer_state_dict': optimizer.state_dict(),  # 옵티마이저 상태
                'val_loss': val_loss,  # 검증 손실
                'val_metrics': val_metrics,  # 검증 메트릭
                'train_metrics': train_metrics  # 학습 메트릭
            }, save_path)
            
            # 모델 저장 알림
            logger.info(f"  → Best model saved! (Val Loss: {val_loss:.4f})")
        else:  # 검증 손실이 개선되지 않은 경우
            patience_counter += 1  # 카운터 증가
        
        # 조기 종료 체크
        if patience_counter >= patience:  # 인내도 초과 시
            logger.info(f"Early stopping at epoch {epoch}")  # 조기 종료 알림
            break  # 학습 루프 종료
    
    # 진행바 닫기
    epoch_pbar.close()
    
    # 학습 결과 반환
    return {
        'best_val_loss': best_val_loss  # 최저 검증 손실
    }


def evaluate_model(
    model,  # 평가할 모델
    test_loader,  # 테스트 데이터 로더
    device,  # 실행 디바이스 (GPU/CPU)
    model_path,  # 저장된 모델 경로
    output_type: str = "target",  # 평가 방식 ("target" 또는 "sequence")
    verbose: bool = True  # 상세 출력 여부
):
    """
    단순화된 모델 평가 함수
    
    Args:
        model: 평가할 모델
        test_loader: 테스트 데이터 로더
        device: 실행 디바이스
        model_path: 모델 경로
        output_type: 평가 방식
            - "target": ref_idx 위치만 평가
            - "sequence": 모든 시퀀스 예측값의 평균 사용
        verbose: 상세 로그 출력 여부
    
    Returns:
        결과 딕셔너리
    """
    
    # 상세 출력 모드일 때 정보 로그
    if verbose:
        logger.info(f"모델 로드: {model_path}")
        logger.info(f"평가 방식: {output_type}")
    
    # 모델 로드
    checkpoint = torch.load(model_path, map_location=device)  # 체크포인트 로드
    model.load_state_dict(checkpoint['model_state_dict'])  # 모델 가중치 로드
    
    # 모델을 디바이스로 이동 및 평가 모드 설정
    model = model.to(device)
    model.eval()  # 평가 모드 (드롭아웃 비활성화, 배치정규화 고정)
    
    # 결과 저장용 리스트 초기화
    all_predictions = []  # 모든 예측값
    all_targets = []  # 모든 실제값
    structured_results = []  # 구조화된 결과 (메타정보 포함)
    
    # 상세 출력 모드일 때 시작 알림
    if verbose:
        logger.info("테스트 시작...")
    
    # 그래디언트 계산 비활성화 (평가 시 메모리 절약)
    with torch.no_grad():
        # 테스트 데이터 배치별 처리
        for batch in tqdm(test_loader, desc="Testing", disable=not verbose):
            # 배치 데이터를 디바이스로 이동
            continuous_data = batch["continuous_data"].to(device)  # 연속형 데이터
            categorical_data = batch["categorical_data"].to(device)  # 범주형 데이터
            position_ids = batch['position_ids'].to(device)  # 위치 ID (window/group 구분)
            reference_oper_groups = batch['reference_oper_groups']  # 참조 그룹
            targets = batch["targets"].to(device)  # 타겟값
            masks = batch["masks"].to(device)  # 유효 데이터 마스크
            sequence_lengths = batch["sequence_lengths"]  # 시퀀스 길이
            
            # 메타 정보 추출
            timekeys = batch.get("timekeys", [])  # 시간 키
            window_oper_ids = batch.get("window_oper_ids", [])  # 윈도우 공정 ID
            
            # 모델 예측 수행
            predictions, adjusted_targets = model(
                continuous_data, categorical_data, position_ids, 
                reference_oper_groups, targets, masks, sequence_lengths
            )
            
            # 텐서를 CPU로 이동하고 NumPy 배열로 변환
            predictions_cpu = predictions.cpu().numpy()
            targets_cpu = adjusted_targets.cpu().numpy()
            
            # 배치 크기 추출
            batch_size = predictions_cpu.shape[0]
            
            # 배치 내 각 샘플 처리
            for b in range(batch_size):
                if output_type == "target":  # 특정 위치(ref_idx)만 평가
                    # predictions_cpu[b]의 차원 확인
                    pred = predictions_cpu[b]  # 현재 샘플의 예측값
                    targ = targets_cpu[b]  # 현재 샘플의 타겟값
                    
                    # 스칼라로 변환 (다양한 차원 처리)
                    if isinstance(pred, np.ndarray):  # NumPy 배열인 경우
                        if pred.ndim == 0:  # 0차원 배열
                            pred_value = float(pred)
                        elif pred.size == 1:  # 1개 요소를 가진 배열
                            pred_value = float(pred.flat[0])
                        else:  # 여러 요소를 가진 경우 첫 번째 값 사용
                            pred_value = float(pred.flat[0])
                    else:  # 이미 스칼라인 경우
                        pred_value = float(pred)
                    
                    # 타겟도 동일하게 처리
                    if isinstance(targ, np.ndarray):
                        if targ.ndim == 0:
                            target_value = float(targ)
                        elif targ.size == 1:
                            target_value = float(targ.flat[0])
                        else:
                            target_value = float(targ.flat[0])
                    else:
                        target_value = float(targ)
                    
                    # 결과 리스트에 추가
                    all_predictions.append(pred_value)
                    all_targets.append(target_value)
                    
                    # 구조화된 결과 저장 (메타정보 포함)
                    structured_results.append({
                        'timekey': timekeys[b] if b < len(timekeys) else None,  # 시간 키
                        'oper_id': window_oper_ids[b][model.ref_idx] if b < len(window_oper_ids) and len(window_oper_ids[b]) > model.ref_idx else None,  # 참조 공정 ID
                        'predicted': pred_value,  # 예측값
                        'actual': target_value  # 실제값
                    })
                    
                elif output_type == "sequence":  # 전체 시퀀스의 평균 사용
                    # 유효한 데이터 마스크 가져오기
                    valid_mask = masks[b].cpu().numpy()
                    
                    # Window 부분만 선택 (position_ids == 0)
                    window_mask = (position_ids[b].cpu().numpy() == 0)
                    # 유효하면서 window 부분인 데이터만 선택
                    valid_window_mask = valid_mask & window_mask
                    
                    if valid_window_mask.any():  # 유효한 window 데이터가 있는 경우
                        # 유효한 window 예측들의 평균 계산
                        valid_preds = predictions_cpu[b][valid_window_mask]
                        valid_targets = targets_cpu[b][valid_window_mask]
                        
                        # 평균값 계산
                        avg_pred = float(np.mean(valid_preds))
                        avg_target = float(np.mean(valid_targets))
                        
                        # 결과 리스트에 추가
                        all_predictions.append(avg_pred)
                        all_targets.append(avg_target)
                        
                        # 구조화된 결과 저장
                        structured_results.append({
                            'timekey': timekeys[b] if b < len(timekeys) else None,  # 시간 키
                            'oper_ids': window_oper_ids[b] if b < len(window_oper_ids) else None,  # 모든 윈도우 공정 ID
                            'predicted': avg_pred,  # 평균 예측값
                            'actual': avg_target,  # 평균 실제값
                            'num_averaged': int(valid_window_mask.sum())  # 평균 계산에 사용된 데이터 수
                        })
    # 메트릭 계산
    metrics = _compute_metrics(all_predictions, all_targets)
    
    # 상세 출력 모드일 때 결과 로그
    if verbose:
        logger.info(f"테스트 결과:")
        logger.info(f"  - RMSE: {metrics['rmse']:.4f}")
        logger.info(f"  - MAE: {metrics['mae']:.4f}")
        logger.info(f"  - MAPE: {metrics['mape']:.2f}%")
        logger.info(f"  - 평가 샘플 수: {len(all_predictions):,}개")
    
    # 평가 결과 딕셔너리 반환
    return {
        "metrics": metrics,  # 계산된 메트릭
        "predictions": all_predictions,  # 모든 예측값
        "targets": all_targets,  # 모든 실제값
        "structured_results": structured_results,  # 구조화된 결과
        "output_type": output_type  # 사용된 평가 방식
    }

def _compute_metrics(predictions, targets, exclude_zeros=True):
    """
    평가 메트릭 계산
    
    Args:
        predictions: 예측값 배열
        targets: 실제값 배열
        exclude_zeros: True면 MAPE/SMAPE 계산 시 타겟 0 제외, 
                      False면 0도 포함 (epsilon으로 처리)
    """
    # 입력을 NumPy 배열로 변환
    predictions = np.array(predictions)
    targets = np.array(targets)
    
    # 기본 메트릭 계산 (모든 데이터 포함)
    mse = np.mean((predictions - targets) ** 2)  # 평균 제곱 오차
    rmse = np.sqrt(mse)  # 평균 제곱근 오차
    mae = np.mean(np.abs(predictions - targets))  # 평균 절대 오차
    
    if exclude_zeros:  # 0 제외 모드
        # 타겟이 0이 아닌 경우만 필터링
        non_zero_mask = targets != 0  # 0이 아닌 위치 마스크
        non_zero_count = non_zero_mask.sum()  # 0이 아닌 데이터 개수
        
        if non_zero_count > 0:  # 0이 아닌 데이터가 있는 경우
            # 0이 아닌 데이터만 추출
            filtered_predictions = predictions[non_zero_mask]
            filtered_targets = targets[non_zero_mask]
            
            # 절대 오차와 절대 타겟값 계산
            abs_errors = np.abs(filtered_predictions - filtered_targets)
            abs_targets = np.abs(filtered_targets)
            
            # MAPE (Mean Absolute Percentage Error) 계산
            mape = np.mean(abs_errors / abs_targets * 100)
            
            # SMAPE (Symmetric Mean Absolute Percentage Error) 계산
            smape = 100 * np.mean(
                2 * abs_errors / (abs_targets + np.abs(filtered_predictions) + 1e-8)
            )
        else:  # 모든 타겟이 0인 경우
            mape = np.nan  # 계산 불가
            smape = np.nan  # 계산 불가
            
    else:  # 0 포함 모드
        # 0도 포함하여 계산 (epsilon 사용)
        epsilon = 1e-8  # 0 나눗셈 방지용 작은 값
        abs_targets = np.abs(targets)  # 타겟 절대값
        abs_errors = np.abs(predictions - targets)  # 오차 절대값
        
        # MAPE - epsilon으로 0 처리
        safe_targets = np.maximum(abs_targets, epsilon)  # 최소값 epsilon 보장
        mape = np.mean(abs_errors / safe_targets * 100)
        
        # SMAPE - epsilon으로 0 처리
        denominator = (abs_targets + np.abs(predictions)) / 2 + epsilon  # 분모 계산
        smape = 100 * np.mean(abs_errors / denominator)
        
        # 0이 아닌 데이터 개수 계산
        non_zero_count = (targets != 0).sum()
    
    # 계산된 모든 메트릭 반환
    return {
        "mse": mse,  # 평균 제곱 오차
        "rmse": rmse,  # 평균 제곱근 오차
        "mae": mae,  # 평균 절대 오차
        "mape": mape,  # 평균 절대 백분율 오차
        "smape": smape,  # 대칭 평균 절대 백분율 오차
        "valid_count": len(predictions),  # 전체 예측 개수
        "non_zero_count": non_zero_count,  # 0이 아닌 타겟 개수
        "zero_ratio": (len(targets) - non_zero_count) / len(targets) * 100,  # 0인 타겟 비율
        "exclude_zeros": exclude_zeros  # 어떤 방식으로 계산했는지 표시
    }

## 🎯 메인 실행

In [10]:
import os
print(f"현재 위치:",os.getcwd())
os.chdir(os.getcwd())

parser = argparse.ArgumentParser(description="시계열 시퀀스 모델링")
parser.add_argument("--config-dir", default="./configs/", help="설정 파일 디렉토리")
parser.add_argument("--mode", choices=["train", "eval"], default="train", help="실행 모드")
parser.add_argument("--model-path", default=None, help="평가용 모델 경로")
parser.add_argument("--gpu", type=int, default=0, help="GPU 번호")
parser.add_argument("--exp-name", default=None, help="실험명")

args = parser.parse_args([])

# 설정 로드
config = load_config(args.config_dir)
set_random_seeds(42)

# 실험명 설정
if args.exp_name:
    exp_name = args.exp_name
else:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    model_type = config.get("model_type", "lstm")
    exp_name = f"{model_type}_{timestamp}"

# 저장 디렉토리
save_dir = config.get("save_dir", "models")
os.makedirs(save_dir, exist_ok=True)
model_save_path = os.path.join(save_dir, f"{exp_name}.pth")

# 디바이스 설정
device = torch.device(f"cuda:{args.gpu}" if torch.cuda.is_available() else "cpu")
logger.info(f"Using device: {device}")

# 데이터로더 생성
logger.info("데이터 로딩 중...")
train_loader, val_loader, test_loader, categorical_processor, ref_idx, unique_groups_num = create_dataloaders(config)

# 모델 생성
vocab_sizes = categorical_processor.get_vocab_sizes()
continuous_dim = len(config["continuous_columns"])

model = create_model(config, vocab_sizes, continuous_dim, ref_idx, unique_groups_num)

if args.mode == "train":
    # 훈련
    logger.info("훈련 시작...")
    train_results = train_model(
        model, train_loader, val_loader, config, device, model_save_path
    )
    
    logger.info("훈련 완료, 테스트 시작...")
    logger.info(f"Evaluation setting: {config['evaluation_setting']}")
    test_results = evaluate_model(model, test_loader, device, model_save_path, output_type=config['evaluation_setting'])
    
else:
    # 평가
    if not args.model_path:
        raise ValueError("--model-path must be provided in eval mode")
    test_results = evaluate_model(model, test_loader, device, args.model_path)

# 결과 저장
results = {
    "exp_name": exp_name,
    "config": config,
    "test_metrics": test_results["metrics"],
    "model_info": {
        "total_parameters": sum(p.numel() for p in model.parameters()),
        "model_type": config.get("model_type", "lstm")
    }
}

# JSON 파일로 결과 저장
results_path = os.path.join(save_dir, f"{exp_name}_results.json")
with open(results_path, "w") as f:
    # JSON 형식으로 저장 (indent=2로 가독성 향상, default=str로 직렬화 불가능한 객체 처리)
    json.dump(results, f, indent=2, default=str)

# 결과 저장 (상세 예측 결과)
# 구조화된 결과가 있는지 확인
if "structured_results" in test_results and test_results["structured_results"]:
    # 구조화된 결과가 있는 경우 (메타정보 포함)
    # DataFrame으로 변환
    structured_df = pd.DataFrame(test_results["structured_results"])
    # CSV 파일 경로 생성
    structured_path = os.path.join(save_dir, f"{exp_name}_structured_predictions.csv")
    # CSV로 저장 (인덱스 제외)
    structured_df.to_csv(structured_path, index=False)
    # 저장 경로 로그 출력
    logger.info(f"구조화된 예측 결과 저장: {structured_path}")
    
    # 요약 통계
    logger.info(f"예측 결과 요약:")
    # 총 예측 수 출력
    logger.info(f"  - 총 예측 수: {len(structured_df)}")
    # timekey 컬럼이 있으면 고유 시간대 수 출력
    if 'timekey' in structured_df.columns:
        logger.info(f"  - 고유한 시간대: {structured_df['timekey'].nunique()}개")
    # oper_id 컬럼이 있으면 고유 공정 수 출력
    if 'oper_id' in structured_df.columns:
        logger.info(f"  - 고유한 oper_id: {structured_df['oper_id'].nunique()}개")
else:
    # 구조화된 정보가 없는 경우 - numpy 배열로 변환
    # 타겟값과 예측값을 NumPy 배열로 변환
    targets = np.array(test_results["targets"])
    predictions = np.array(test_results["predictions"])
    
    # 예측 결과 DataFrame 생성
    predictions_df = pd.DataFrame({
        "actual": targets,  # 실제값
        "predicted": predictions,  # 예측값
        "residual": targets - predictions,  # 잔차 (실제값 - 예측값)
        "abs_error": np.abs(targets - predictions),  # 절대 오차
        "abs_percent_error": (  # 절대 백분율 오차
            np.abs(targets - predictions) /  # 절대 오차를
            np.maximum(np.abs(targets), 1e-8) * 100  # 타겟 절대값으로 나누고 100 곱함 (0 방지)
        )
    })
    
    # CSV 파일 경로 생성
    predictions_path = os.path.join(save_dir, f"{exp_name}_predictions.csv")
    # CSV로 저장 (인덱스 제외)
    predictions_df.to_csv(predictions_path, index=False)
    # 저장 경로 로그 출력
    logger.info(f"예측 결과 저장: {predictions_path}")

현재 위치: /home/doyoon/teaching/SKhynix_2025/SKhynix_week5/opertransformer


2025-09-13 06:33:35,834 - INFO - Using device: cuda:0
2025-09-13 06:33:35,835 - INFO - 데이터 로딩 중...
2025-09-13 06:48:44,637 - INFO - 데이터 전처리 시작...
2025-09-13 06:48:45,574 - WARNING - Inf 값 발견! 제거합니다.
2025-09-13 06:48:47,571 - INFO - 범주형 변수별 고유값 개수:
2025-09-13 06:48:47,571 - INFO -   oper_group: 277개
2025-09-13 06:48:47,572 - INFO -   days: 7개
2025-09-13 06:48:47,573 - INFO -   shift: 3개
2025-09-13 06:48:47,573 - INFO -   x1: 20개
2025-09-13 06:48:52,213 - INFO - 날짜 기준 데이터 분할 완료:
2025-09-13 06:48:52,214 - INFO -   - 총 날짜 수: 90일
2025-09-13 06:48:52,214 - INFO -   - Train: 72일 (1,342,808행)
2025-09-13 06:48:52,215 - INFO -   - Validation: 9일 (170,705행)
2025-09-13 06:48:52,215 - INFO -   - Test: 9일 (157,944행)
2025-09-13 06:48:52,216 - INFO -   - Train 날짜 범위: 20250503 ~ 20250713
2025-09-13 06:48:52,216 - INFO -   - Val 날짜 범위: 20250714 ~ 20250722
2025-09-13 06:48:52,217 - INFO -   - Test 날짜 범위: 20250723 ~ 20250731
2025-09-13 06:48:52,217 - INFO - 연속형 변수 정규화
2025-09-13 06:48:52,218 - INFO - Stan